# FINA4030A — Augmented Intelligence in Finance
### Environment check

**Three minutes. Not graded.**

This notebook confirms that your laptop can run the labs, and tells you what to do if it can't.
We would much rather find a problem now than in the middle of the first lab on **Thursday 10 September**.

**What to do**

1. Run the cell below (`Shift`+`Enter`, or the ▶ button to its left). First, put your name and student ID between the quotes at the top of it.
2. Wait for it to finish — a few seconds.
3. Screenshot **everything it prints, including the chart**, and submit the screenshot on Blackboard.

You do not need to install anything, and nothing here costs money. If the cell reports warnings, that is fine and expected — read the note underneath it.


In [ ]:
# ============================================================================
#  FINA4030A  Augmented Intelligence in Finance
#  Environment check  --  run this cell, then screenshot the output
#
#  1. Put your name and student ID between the quotes below.
#  2. Run this cell (Shift+Enter, or the play button on the left).
#  3. Screenshot everything it prints, including the chart, and submit it.
#
#  This is not graded. It exists so that we find problems now rather than in
#  the middle of the first lab.
# ============================================================================

NAME       = ""          # e.g. "CHAN Tai Man"
STUDENT_ID = ""          # e.g. "1155123456"

# ---------------------------------------------------------------- (no need to
# ---------------------------------------------------------------- edit below)

import sys, os, platform, hashlib, importlib, importlib.util, io, textwrap
from datetime import datetime, timezone

CHECKS, LIBS = [], []
def rec(store, status, label, detail=""):
    store.append((status, label, detail))

# ---------- who and when -----------------------------------------------------
try:
    from zoneinfo import ZoneInfo
    now = datetime.now(ZoneInfo("Asia/Hong_Kong")); tzname = "Asia/Hong_Kong"
except Exception:
    now = datetime.now(timezone.utc); tzname = "UTC"

who = (NAME.strip() or "[NAME NOT FILLED IN]")
sid = (STUDENT_ID.strip() or "[ID NOT FILLED IN]")
code = hashlib.sha256(f"{who}|{sid}".encode()).hexdigest()[:8].upper()
vcode = f"{code[:4]}-{code[4:]}"

# ---------- runtime ----------------------------------------------------------
if "google.colab" in sys.modules or importlib.util.find_spec("google.colab"):
    runtime = "Google Colab"
elif sys.platform == "emscripten":
    runtime = "Browser (JupyterLite / Pyodide)"
else:
    runtime = "Local Python / other"

def ram_gb():
    try:
        import psutil; return f"{psutil.virtual_memory().total/2**30:.1f} GB"
    except Exception:
        pass
    try:
        with open("/proc/meminfo") as f:
            for line in f:
                if line.startswith("MemTotal"):
                    return f"{int(line.split()[1])/2**20:.1f} GB"
    except Exception:
        pass
    return "unknown"

# ---------- libraries --------------------------------------------------------
def check_lib(name, required=True):
    try:
        m = importlib.import_module(name)
        rec(LIBS, "OK", name, getattr(m, "__version__", "present"))
        return m
    except Exception as e:
        rec(LIBS, "FAIL" if required else "WARN", name,
            "not available" if isinstance(e, ImportError) else str(e)[:50])
        return None

pd  = check_lib("pandas")
np  = check_lib("numpy")
mpl = check_lib("matplotlib")
pa  = check_lib("pyarrow", required=False)

# ---------- functional checks ------------------------------------------------
df = None
if np is not None:
    try:
        # RandomState, not default_rng: NumPy guarantees this stream is identical
        # across versions, so every student's numbers must match. default_rng
        # carries no such guarantee. We return to this point in Class 2.
        rng = np.random.RandomState(4030)
        rets = rng.normal(0.0004, 0.011, 250)
        rec(CHECKS, "OK", "seeded returns", f"mean = {rets.mean():+.8f}  (must match everyone)")
    except Exception as e:
        rec(CHECKS, "FAIL", "numpy arithmetic", str(e)[:60]); rets = None
else:
    rets = None

if pd is not None and rets is not None:
    try:
        idx = pd.date_range("2025-01-01", periods=250, freq="B")
        df = pd.DataFrame({"ret": rets}, index=idx)
        df["level"] = 100 * (1 + df["ret"]).cumprod()
        df["drawdown"] = df["level"] / df["level"].cummax() - 1
        rec(CHECKS, "OK", "pandas dataframe", f"{df.shape[0]} rows x {df.shape[1]} cols")
    except Exception as e:
        rec(CHECKS, "FAIL", "pandas dataframe", str(e)[:60])

if df is not None:
    if pa is not None:
        try:
            buf = io.BytesIO(); df.to_parquet(buf); buf.seek(0)
            back = pd.read_parquet(buf)
            ok = len(back) == len(df)
            rec(CHECKS, "OK" if ok else "FAIL", "parquet round-trip",
                f"wrote and read {len(back)} rows")
        except Exception as e:
            rec(CHECKS, "WARN", "parquet round-trip", str(e)[:60])
    else:
        try:
            buf = io.StringIO(); df.to_csv(buf); buf.seek(0)
            back = pd.read_csv(buf, index_col=0)
            rec(CHECKS, "WARN", "parquet round-trip",
                f"skipped, no pyarrow; CSV fallback read {len(back)} rows")
        except Exception as e:
            rec(CHECKS, "FAIL", "csv fallback", str(e)[:60])

fig = None
if mpl is not None and df is not None:
    try:
        import matplotlib
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(1, 2, figsize=(9.5, 3.1))
        ax[0].plot(df.index, df["level"], lw=1.3, color="#1f3b73")
        ax[0].set_title("Seeded price path (identical for everyone)", fontsize=9)
        ax[0].tick_params(labelsize=7); ax[0].grid(alpha=.25)
        ax[1].fill_between(df.index, df["drawdown"], 0, color="#b03a2e", alpha=.7, lw=0)
        ax[1].set_title("Drawdown", fontsize=9)
        ax[1].tick_params(labelsize=7); ax[1].grid(alpha=.25)
        for a in ax:
            for lbl in a.get_xticklabels(): lbl.set_rotation(20)
        fig.suptitle(f"FINA4030A environment check  ·  {who}  ·  {vcode}", fontsize=9.5)
        fig.tight_layout()
        rec(CHECKS, "OK", "matplotlib render", f"backend {matplotlib.get_backend()}")
    except Exception as e:
        rec(CHECKS, "FAIL", "matplotlib render", str(e)[:60])

# ---------- report -----------------------------------------------------------
BAR = "=" * 74
def line(status, label, detail):
    tag = {"OK": "[ OK ]", "WARN": "[WARN]", "FAIL": "[FAIL]"}[status]
    return f"  {tag}  {label:<22} {detail}"

out = [BAR, "  FINA4030A  Environment check", BAR,
       f"  Student            {who}  ({sid})",
       f"  Run at             {now:%Y-%m-%d %H:%M} ({tzname})",
       f"  Verification code  {vcode}", "",
       "  ENVIRONMENT",
       f"         Runtime            {runtime}",
       f"         Python             {sys.version.split()[0]}",
       f"         Platform           {platform.system()} {platform.release()} / {platform.machine()}",
       f"         Memory             {ram_gb()}",
       f"         CPU cores          {os.cpu_count()}", "",
       "  LIBRARIES"]
out += [line(s, l, d) for s, l, d in LIBS]
out += ["", "  CHECKS"] + [line(s, l, d) for s, l, d in CHECKS]

fails = sum(1 for s, _, _ in LIBS + CHECKS if s == "FAIL")
warns = sum(1 for s, _, _ in LIBS + CHECKS if s == "WARN")
missing_name = who.startswith("[") or sid.startswith("[")

if fails:
    verdict, msg = "NOT READY", "Email the instructor a screenshot of this output."
elif missing_name:
    verdict, msg = "READY (but fill in your name)", "Add your name and ID above, then run again."
else:
    verdict, msg = "READY", "Screenshot this output and the chart below, then submit."

out += ["", BAR,
        f"  RESULT: {verdict}   ({fails} failure(s), {warns} warning(s))",
        f"  {msg}", BAR]
print("\n".join(out))

if warns and not fails:
    print(textwrap.dedent("""
      A warning is not a problem. Warnings mean an optional component is absent
      and the course will use a fallback. You do not need to install anything.
    """).rstrip())

if fig is not None:
    import matplotlib.pyplot as plt
    plt.show()

---

## Reading your result

| Result | What it means | What to do |
|---|---|---|
| **READY**, 0 failures | Everything the labs need is working. | Screenshot and submit. Done. |
| **READY**, some warnings | An optional component is missing and the course will use a fallback. | Screenshot and submit. Nothing to fix. |
| **READY (but fill in your name)** | The checks passed but the name field is empty. | Add your name and ID at the top of the cell, run it again, then screenshot. |
| **NOT READY** | Something the labs need is genuinely broken. | Email me the screenshot. Do not spend time fighting it. |

## If it did not work at all

**The cell errored immediately, or nothing happened.** Make sure you opened this notebook in the environment named in the setup announcement rather than downloading the file and opening it in something else. If you are signed in and it still fails, send me a screenshot.

**You are on a tablet.** Some of this may not run. That is a known limitation, not your fault — tell me in the pre-course survey and I will pair you with someone for the labs.

**Everything is in a language you did not expect, or the chart is missing.** Screenshot it anyway and send it. An odd result is useful information for me.

**Something took longer than about thirty seconds.** That is worth reporting too. The labs assume a session starts quickly.

## One thing worth noticing

The chart is generated from a *seeded* random sequence, which means **every student in the class should get an identical chart and an identical mean**. Compare with a classmate if you like.

That is a deliberate choice, and not the default behaviour. Reproducing a result exactly — knowing which parts of your work are pinned and which are not — is one of the things this course is about. We come back to it in Class 2, where you will find that the tools you are about to rely on do *not* give you this guarantee.

---

## Optional: network check

Only run the cell below if you have a spare minute. The labs use data files supplied in advance, so you do **not** need this to pass, and a failure here is not a problem.


In [ ]:
# Optional. A failure here does not matter for the labs.
DATA_URL = "https://api.hkma.gov.hk/public/market-data-and-statistics/daily-monetary-statistics/daily-figures-interbank-liquidity?pagesize=1"

import sys, json, urllib.request, time

print("Network check — optional\n" + "-" * 46)
if sys.platform == "emscripten":
    print("  Browser runtime detected.")
    print("  Direct network calls are blocked here by browser security.")
    print("  This is expected and does not affect the labs, which use")
    print("  data files supplied in advance. Nothing to do.")
else:
    try:
        t0 = time.time()
        req = urllib.request.Request(DATA_URL, headers={"User-Agent": "FINA4030A setup check"})
        with urllib.request.urlopen(req, timeout=20) as r:
            payload = json.loads(r.read().decode("utf-8"))
        dt = time.time() - t0
        ok = isinstance(payload, dict) and "result" in payload
        print(f"  [ OK ]  reached the data source in {dt:.1f}s")
        print(f"          response {'has the expected HKMA structure' if ok else 'was JSON but an unexpected shape'}")
        print("\n  Good — you can reach public financial data directly.")
    except Exception as e:
        print(f"  [WARN]  could not reach the data source: {type(e).__name__}")
        print(f"          {str(e)[:70]}")
        print("\n  Not a problem. The labs use pre-supplied data files.")
        print("  Mention it in the pre-course survey if you are curious why.")

---

That is everything. See you on **Thursday 10 September, 14:30, HYS G04**.

Eric Lam · fyericlam@cuhk.edu.hk · CYT 854 · office hours Monday 11:30–13:30
